In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.optimize import minimize
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# File paths (relative to notebook location)
data_dir = Path("../data")

mv_monthly_file = data_dir / "DS_MV_T_USD_M_2025.xlsx"
mv_yearly_file  = data_dir / "DS_MV_T_USD_Y_2025.xlsx"
rev_file        = data_dir / "DS_REV_Y_2025.xlsx"
ri_monthly_file = data_dir / "DS_RI_T_USD_M_2025.xlsx"
ri_yearly_file  = data_dir / "DS_RI_T_USD_Y_2025.xlsx"
rf_file         = data_dir / "Risk_Free_Rate_2025.xlsx"
scope1_file     = data_dir / "DS_CO2_SCOPE_1_Y_2025.xlsx"
scope2_file     = data_dir / "DS_CO2_SCOPE_2_Y_2025.xlsx"
static_file     = data_dir / "Static_2025.xlsx"

# Load datasets
mcap_monthly = pd.read_excel(mv_monthly_file)
mcap_yearly  = pd.read_excel(mv_yearly_file)
revenue      = pd.read_excel(rev_file)
price_monthly = pd.read_excel(ri_monthly_file)
price_yearly  = pd.read_excel(ri_yearly_file)
risk_free    = pd.read_excel(rf_file)
scope1       = pd.read_excel(scope1_file)
scope2       = pd.read_excel(scope2_file)
static_data  = pd.read_excel(static_file)

# Keep only firms from North America and Europe (Group J)
static_data["Region"] = static_data["Region"].str.strip()
static_data = static_data[static_data["Region"].isin(["AMER", "EUR"])]

print("Regions kept:", static_data["Region"].unique())
print("Filtered dataset shape:", static_data.shape)

# Result : After filtering the dataset based on Datastream regional codes (AMER and EUR), the investment universe consists of 1302 firms located in North America and Europe.

# Convert the risk-free rate date column to datetime
risk_free["Date"] = pd.to_datetime(risk_free["Unnamed: 0"].astype(str), format="%Y%m")
risk_free = risk_free.set_index("Date")

# Keep only firms belonging to the Group J investment universe
mcap_monthly = mcap_monthly[mcap_monthly["ISIN"].isin(static_data["ISIN"])]
mcap_yearly  = mcap_yearly[mcap_yearly["ISIN"].isin(static_data["ISIN"])]
revenue      = revenue[revenue["ISIN"].isin(static_data["ISIN"])]
price_monthly = price_monthly[price_monthly["ISIN"].isin(static_data["ISIN"])]
price_yearly  = price_yearly[price_yearly["ISIN"].isin(static_data["ISIN"])]
scope1       = scope1[scope1["ISIN"].isin(static_data["ISIN"])]
scope2       = scope2[scope2["ISIN"].isin(static_data["ISIN"])]

# Quick checks
print("Risk-free rate shape:", risk_free.shape)
print("Filtered monthly market cap shape:", mcap_monthly.shape)
print("Filtered yearly market cap shape:", mcap_yearly.shape)
print("Filtered revenue shape:", revenue.shape)
print("Filtered monthly price shape:", price_monthly.shape)
print("Filtered yearly price shape:", price_yearly.shape)
print("Filtered Scope 1 shape:", scope1.shape)
print("Filtered Scope 2 shape:", scope2.shape)


Regions kept: <StringArray>
['AMER', 'EUR']
Length: 2, dtype: str
Filtered dataset shape: (1302, 4)
Risk-free rate shape: (312, 2)
Filtered monthly market cap shape: (1302, 316)
Filtered yearly market cap shape: (1302, 29)
Filtered revenue shape: (1302, 29)
Filtered monthly price shape: (1302, 316)
Filtered yearly price shape: (1302, 29)
Filtered Scope 1 shape: (1302, 29)
Filtered Scope 2 shape: (1302, 29)


In [24]:
# Create a dictionary to map ISIN codes to firm names
isin_to_name = dict(zip(static_data["ISIN"], static_data["NAME"]))

# Drop the NAME column where it exists
mcap_monthly = mcap_monthly.drop(columns=["NAME"], errors="ignore")
mcap_yearly  = mcap_yearly.drop(columns=["NAME"], errors="ignore")
revenue      = revenue.drop(columns=["NAME"], errors="ignore")
price_monthly = price_monthly.drop(columns=["NAME"], errors="ignore")
price_yearly  = price_yearly.drop(columns=["NAME"], errors="ignore")
scope1       = scope1.drop(columns=["NAME"], errors="ignore")
scope2       = scope2.drop(columns=["NAME"], errors="ignore")

# Set ISIN as index
mcap_monthly = mcap_monthly.set_index("ISIN")
mcap_yearly  = mcap_yearly.set_index("ISIN")
revenue      = revenue.set_index("ISIN")
price_monthly = price_monthly.set_index("ISIN")
price_yearly  = price_yearly.set_index("ISIN")
scope1       = scope1.set_index("ISIN")
scope2       = scope2.set_index("ISIN")

# Transpose datasets so that dates become the index
mcap_monthly = mcap_monthly.T.sort_index(ascending=True)
mcap_yearly  = mcap_yearly.T.sort_index(ascending=True)
revenue      = revenue.T.sort_index(ascending=True)
price_monthly = price_monthly.T.sort_index(ascending=True)
price_yearly  = price_yearly.T.sort_index(ascending=True)
scope1       = scope1.T.sort_index(ascending=True)
scope2       = scope2.T.sort_index(ascending=True)

print(price_monthly.head(10))

ISIN                AN8068571086 AT000000STR1 AT00000VIE62 AT0000606306 AT0000652011 AT0000720008 AT0000730007  \
1999-12-31 00:00:00      1708.01          NaN       147.79          NaN       102.94          NaN          NaN   
2000-01-31 00:00:00      1858.26          NaN       156.25          NaN        94.91          NaN          NaN   
2000-02-29 00:00:00      2254.15          NaN       153.83          NaN        97.74          NaN          NaN   
2000-03-31 00:00:00      2334.75          NaN       158.62          NaN        100.5          NaN          NaN   
2000-04-28 00:00:00      2336.65          NaN       137.27          NaN        96.13          NaN          NaN   
2000-05-31 00:00:00      2245.09          NaN       148.95          NaN        97.86          NaN          NaN   
2000-06-30 00:00:00       2283.7          NaN       159.68          NaN       102.06          NaN          NaN   
2000-07-31 00:00:00      2262.67          NaN       151.08          NaN       102.43    

In [25]:
def process_interior_missing_values(data: pd.DataFrame) -> dict:
    """
    Process only interior missing values in each column of a time-series DataFrame.

    Rules:
    - Missing values before the first valid observation are left untouched.
    - Missing values after the last valid observation are left untouched.
    - A single missing value inside the valid span is linearly interpolated.
    - Multiple consecutive missing values are recorded in 'targeted' and not filled.

    Parameters
    ----------
    data : pd.DataFrame
        Time-series DataFrame with dates as index and firms as columns.

    Returns
    -------
    dict
        A dictionary containing:
        - 'interpolated_df': the cleaned DataFrame
        - 'targeted': columns and gap locations with multiple consecutive NaNs
        - 'interpolated_columns': list of columns where isolated NaNs were filled
    """
    df_interpolated = data.copy()
    targeted = {}
    interpolated_columns = []

    for col in data.columns:
        series = data[col]

        first_valid = series.first_valid_index()
        last_valid = series.last_valid_index()

        # Skip empty or trivial series
        if first_valid is None or last_valid is None or first_valid == last_valid:
            continue

        # Focus only on the interior part of the series
        interior_series = series.loc[first_valid:last_valid]

        sequences = []
        current_seq_start = None
        current_seq_length = 0

        # Detect consecutive NaN sequences
        for idx, val in interior_series.items():
            if pd.isna(val):
                if current_seq_start is None:
                    current_seq_start = idx
                    current_seq_length = 1
                else:
                    current_seq_length += 1
            else:
                if current_seq_start is not None:
                    sequences.append((current_seq_start, idx, current_seq_length))
                    current_seq_start = None
                    current_seq_length = 0

        # If the series ends with an interior missing block
        if current_seq_start is not None:
            sequences.append((current_seq_start, interior_series.index[-1], current_seq_length))

        gap_multiple = False

        for seq in sequences:
            start_seq, end_seq, length = seq
            if length > 1:
                gap_multiple = True
                if col not in targeted:
                    targeted[col] = []
                targeted[col].append(seq)

        # Interpolate only if all interior gaps are isolated single NaNs
        if not gap_multiple and len(sequences) > 0:
            df_interpolated.loc[first_valid:last_valid, col] = (
                df_interpolated.loc[first_valid:last_valid, col]
                .interpolate(method="linear", limit=1, limit_direction="both")
            )
            interpolated_columns.append(col)

    if len(targeted) == 0:
        print("No multiple interior gaps were detected.")
        print("Isolated missing values were interpolated when present.")
    else:
        print("Multiple interior gaps were detected in the following columns:")
        print(targeted)

    return {
        "interpolated_df": df_interpolated,
        "targeted": targeted,
        "interpolated_columns": interpolated_columns
    }

price_missing_results = process_interior_missing_values(price_monthly)
price_monthly_clean = price_missing_results["interpolated_df"]

print("NaN before:", price_monthly.isna().sum().sum())
print("NaN after :", price_monthly_clean.isna().sum().sum())
print("Columns interpolated:", len(price_missing_results["interpolated_columns"]))
print("Columns with large gaps:", len(price_missing_results["targeted"]))

No multiple interior gaps were detected.
Isolated missing values were interpolated when present.
NaN before: 17082
NaN after : 17082
Columns interpolated: 0
Columns with large gaps: 0


In [26]:
def generate_windows_with_returns(
    price_data: pd.DataFrame,
    first_test_year: int = 2014,
    last_test_year: int = 2025,
    estimation_months: int = 120,
    min_months_available: int = 36
) -> dict:
    """
    Generate rolling estimation and test windows from monthly price data.

    Parameters
    ----------
    price_data : pd.DataFrame
        Monthly price DataFrame with dates as index and ISINs as columns.
    first_test_year : int
        First out-of-sample year (default: 2014).
    last_test_year : int
        Last out-of-sample year (default: 2025).
    estimation_months : int
        Number of monthly observations used for estimation (default: 120).
    min_months_available : int
        Minimum number of valid monthly returns required to keep a firm eligible
        in the estimation window (default: 36).

    Returns
    -------
    dict
        Dictionary indexed by test year. Each entry contains:
        - estimation_prices
        - test_prices
        - estimation_returns
        - test_returns
        - eligible_companies
        - excluded_companies
        - failure_count
    """
    df = price_data.copy()
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()

    # Monthly simple returns
    returns = df.pct_change(fill_method=None)

    windows = {}

    for year in range(first_test_year, last_test_year + 1):
        test_start = pd.Timestamp(f"{year}-01-01")
        test_end = pd.Timestamp(f"{year}-12-31")

        # Estimation window ends the month before the test period starts
        estimation_end = test_start - pd.offsets.MonthEnd(1)

        # Start date so that we keep exactly 120 months if available
        estimation_start = estimation_end - pd.DateOffset(months=estimation_months - 1)

        # Slice prices
        estimation_prices = df.loc[(df.index >= estimation_start) & (df.index <= estimation_end)]
        test_prices = df.loc[(df.index >= test_start) & (df.index <= test_end)]

        # Slice returns
        estimation_returns = returns.loc[(returns.index >= estimation_start) & (returns.index <= estimation_end)]
        test_returns = returns.loc[(returns.index >= test_start) & (returns.index <= test_end)]

        # Skip empty windows
        if estimation_returns.empty or test_returns.empty:
            continue

        # Keep only firms with enough return observations in the estimation period
        eligible_companies = []
        excluded_companies = []

        for col in estimation_returns.columns:
            if estimation_returns[col].count() >= min_months_available:
                eligible_companies.append(col)
            else:
                excluded_companies.append(col)

        estimation_prices = estimation_prices[eligible_companies]
        test_prices = test_prices[eligible_companies]
        estimation_returns = estimation_returns[eligible_companies]
        test_returns = test_returns[eligible_companies]

        if estimation_returns.empty or test_returns.empty:
            continue

        # Count firms with at least one missing return during the test year
        failure_count = sum(test_returns[col].isna().any() for col in test_returns.columns)

        windows[str(year)] = {
            "estimation_prices": estimation_prices,
            "test_prices": test_prices,
            "estimation_returns": estimation_returns,
            "test_returns": test_returns,
            "eligible_companies": eligible_companies,
            "excluded_companies": excluded_companies,
            "failure_count": failure_count,
            "estimation_start": estimation_start,
            "estimation_end": estimation_end,
            "test_start": test_start,
            "test_end": test_end
        }

    return windows